# University Chapters Data Product
# Cell-1

## Project Objective
Build an Azure-style Medallion Data Product using Databricks and Spark.

### Pipeline
API → Bronze → Silver → Data Quality → Quarantine → Gold

### Source
ArcGIS University Chapters Public API

### Scope
CA, OR, WA

In [0]:
%python
import requests
print("python environment is working")

Cell 2 — Define the API URL

In [0]:
%python
API_URL = "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/UniversityChapters_Public/FeatureServer/0/query"

print(API_URL)


Cell 3 - Instruct API to send the details

In [0]:
%python
params = {
    "where": "State IN ('CA','OR','WA')",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "json"
}

print(params)

 Cell 4 - Call the API

In [0]:
%python
response = requests.get(
    API_URL,
    params=params,
    timeout=60
)

print("HTTP Status:", response.status_code)


Cell 5 — Convert API response into JSON

In [0]:
%python
api_response = response.json()

print(type(api_response))


Cell 6 — Get the records

In [0]:
%python
features = api_response.get("features", [])

# API validation

if response.status_code != 200:
    raise RuntimeError(
        f"API request failed with HTTP status {response.status_code}"
    )

if "error" in api_response:
    raise RuntimeError(
        f"API returned an error: {api_response['error']}"
    )

if not features:
    raise RuntimeError(
        "Source returned zero records for CA/OR/WA."
    )

# Check state distribution
state_counts = {}

for feature in features:
    state = feature.get("attributes", {}).get("State")
    state_counts[state] = state_counts.get(state, 0) + 1

# CA is expected to contain data
if state_counts.get("CA", 0) == 0:
    raise RuntimeError(
        "CA unexpectedly contains zero records."
    )

print("API validation passed.")
print("State counts:", state_counts)

print("Number of records received:", len(features))

Inspect one API record

In [0]:
%python
import json

print(json.dumps(features[0], indent=2))

Create our first Bronze record

In [0]:
%python
from datetime import datetime, timezone
import json

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGEST_TIMESTAMP = datetime.now(timezone.utc).isoformat()

print("Run ID:", RUN_ID)
print("Ingest timestamp:", INGEST_TIMESTAMP)

Create the Bronze dataset in memory

In [0]:
%python
bronze_records = []

for feature in features:
    bronze_records.append({
        "ingest_run_id": RUN_ID,
        "ingest_timestamp": INGEST_TIMESTAMP,
        "source_name": "UniversityChapters_Public",
        "source_url": API_URL,
        "raw_payload": json.dumps(feature)
    })

print("Bronze records prepared:", len(bronze_records))
print(bronze_records[0])

Check which states we received

In [0]:
%python
for feature in features:
    attributes = feature.get("attributes", {})
    print(
        "ChapterID:", attributes.get("ChapterID"),
        "| State:", attributes.get("State"),
        "| City:", attributes.get("City")
    )

Let's check the distribution

In [0]:
%python
state_counts = {}

for feature in features:
    state = feature.get("attributes", {}).get("State")
    state_counts[state] = state_counts.get(state, 0) + 1

print(state_counts)

Create a Bronze DataFrame

In [0]:
%python
bronze_df = spark.createDataFrame(bronze_records)

display(bronze_df)

Bronze location

In [0]:
%python
# create catalog
# CREATE SCHEMA IF NOT EXISTS university_chapters.data_product;
# SHOW SCHEMAS IN university_chapters;
# CREATE VOLUME IF NOT EXISTS university_chapters.data_product.project_files;

#display(dbutils.fs.ls("/Volumes/university_chapters/data_product/project_files/"))
BASE_PATH = "/Volumes/university_chapters/data_product/project_files"

display(dbutils.fs.ls(BASE_PATH))

Create Project Folders

In [0]:
%python
BRONZE_PATH = f"{BASE_PATH}/bronze/university_chapters"
SILVER_PATH = f"{BASE_PATH}/silver/university_chapters"
GOLD_PATH = f"{BASE_PATH}/gold/university_chapters/v1"
QUARANTINE_PATH = f"{BASE_PATH}/quarantine/university_chapters"

for path in [
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    QUARANTINE_PATH
]:
    dbutils.fs.mkdirs(path)

print("Project folders created successfully!")

Now save the bronze

In [0]:
%python
bronze_run_path = f"{BRONZE_PATH}/{RUN_ID}"

bronze_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(bronze_run_path)

print("Bronze data written to:")
print(bronze_run_path)

Verify the Bronze exists 

In [0]:
%python
display(dbutils.fs.ls(bronze_run_path))


Read bronze back

In [0]:
%python
bronze_read_df = spark.read.parquet(bronze_run_path)

display(bronze_read_df)

Parse JSON into Spark columns

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw_schema = T.StructType([
    T.StructField(
        "attributes",
        T.StructType([
            T.StructField("ChapterID", T.StringType(), True),
            T.StructField("University_Chapter", T.StringType(), True),
            T.StructField("City", T.StringType(), True),
            T.StructField("State", T.StringType(), True),
            T.StructField("OBJECTID", T.StringType(), True)
        ]),
        True
    ),
    T.StructField(
        "geometry",
        T.StructType([
            T.StructField("x", T.DoubleType(), True),
            T.StructField("y", T.DoubleType(), True)
        ]),
        True
    )
])

display(raw_schema)

Parse the JSON

In [0]:
%python
silver_source_df = (
    bronze_read_df
    .withColumn(
        "payload",
        F.from_json(F.col("raw_payload"), raw_schema)
    )
)

display(silver_source_df.select("payload"))

Create the Silver columns

In [0]:
%python
silver_df = (
    silver_source_df
    .select(
        F.col("payload.attributes.ChapterID")
            .cast("string")
            .alias("chapter_id"),

        F.trim(
            F.col("payload.attributes.University_Chapter")
        ).alias("chapter_name"),

        F.trim(
            F.col("payload.attributes.City")
        ).alias("city"),

        F.upper(
            F.trim(
                F.col("payload.attributes.State")
            )
        ).alias("state"),

        F.col("payload.geometry.x")
            .cast("double")
            .alias("longitude"),

        F.col("payload.geometry.y")
            .cast("double")
            .alias("latitude"),

        F.col("payload.attributes.OBJECTID")
            .alias("source_object_id"),

        F.col("ingest_run_id"),
        F.col("ingest_timestamp"),
        F.col("raw_payload")
    )
)

display(silver_df)

First, deduplicate Silver

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

dedupe_window = (
    Window
    .partitionBy("chapter_id")
    .orderBy(
        F.col("ingest_timestamp").desc(),
        F.col("source_object_id").desc()
    )
)

silver_dedup_df = (
    silver_df
    .withColumn("row_number", F.row_number().over(dedupe_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

display(silver_dedup_df)

print("Before deduplication:", silver_df.count())
print("After deduplication:", silver_dedup_df.count())

Create the invalid-coordinate condition and
Identify the bad records

In [0]:
%python
coordinate_invalid_condition = (
    F.col("longitude").isNull()
    | F.col("latitude").isNull()
    | (F.col("longitude") < -180)
    | (F.col("longitude") > 180)
    | (F.col("latitude") < -90)
    | (F.col("latitude") > 90)
)

invalid_coordinates_df = (
    silver_dedup_df
    .filter(coordinate_invalid_condition)
)

display(coordinate_invalid_condition)
display(invalid_coordinates_df)

Create valid Silver records

In [0]:
%python
silver_valid_df = (
    silver_dedup_df
    .filter(~coordinate_invalid_condition)
)

display(silver_valid_df)

Create the Quarantine dataset

In [0]:
%python
quarantine_df = (
    invalid_coordinates_df
    .withColumn(
        "dq_reason_code",
        F.lit("INVALID_COORDINATES")
    )
    .withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )
    .select(
        "ingest_run_id",
        "chapter_id",
        "chapter_name",
        "city",
        "state",
        "longitude",
        "latitude",
        "dq_reason_code",
        "quarantine_timestamp",
        "raw_payload"
    )
)
display(quarantine_df)

Save quarantine

In [0]:
%python
quarantine_run_path = f"{QUARANTINE_PATH}/{RUN_ID}"

quarantine_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(quarantine_run_path)

print("Quarantine saved to:")
print(quarantine_run_path)

Create the city check and update the column values and save silver

In [0]:
%python
missing_city_condition = (
    F.col("city").isNull()
    | (F.trim(F.col("city")) == "")
    | (F.upper(F.trim(F.col("city"))) == "UNKNOWN")
)

silver_with_dq_df = (
    silver_valid_df
    .withColumn(
        "dq_status",
        F.when(
            missing_city_condition,
            F.lit("WARNING")
        ).otherwise(
            F.lit("OK")
        )
    )
    .withColumn(
        "dq_warnings",
        F.when(
            missing_city_condition,
            F.array(F.lit("MISSING_OR_UNKNOWN_CITY"))
        ).otherwise(
            F.array().cast("array<string>")
        )
    )
)

display(silver_with_dq_df)

#Save Silver
silver_with_dq_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(SILVER_PATH)

print("Silver data saved successfully:")
print(SILVER_PATH)

# also verify read 
silver_read_df = spark.read.parquet(SILVER_PATH)

display(silver_read_df)
silver_read_df.printSchema()

Now Create the Audit records and save it

In [0]:
%python
# calculate the counts 
rows_in = silver_dedup_df.count()

rows_quarantined = quarantine_df.count()

rows_warned = (
    silver_with_dq_df
    .filter(F.col("dq_status") == "WARNING")
    .count()
)

rows_ok = (
    silver_with_dq_df
    .filter(F.col("dq_status") == "OK")
    .count()
)

print("Rows in:", rows_in)
print("Rows quarantined:", rows_quarantined)
print("Rows warned:", rows_warned)
print("Rows OK:", rows_ok)

# Create the audit record
audit_df = spark.createDataFrame(
    [
        (
            RUN_ID,
            rows_in,
            rows_quarantined,
            rows_warned,
            rows_ok
        )
    ],
    [
        "ingest_run_id",
        "rows_in",
        "rows_quarantined",
        "rows_warned",
        "rows_ok"
    ]
)

display(audit_df)

# save the audit information
AUDIT_PATH = f"{BASE_PATH}/audit/university_chapters"
audit_df.write \
    .mode("append") \
    .format("parquet") \
    .save(AUDIT_PATH)

print("Audit saved to:",AUDIT_PATH)


Create the Gold Data and save it 

In [0]:
%python
gold_df = (
    silver_with_dq_df
    .select(
        "chapter_id",
        "chapter_name",
        "city",
        "state",
        "longitude",
        "latitude",
        "dq_status",
        "dq_warnings"
    )
)

display(gold_df)

gold_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(GOLD_PATH)

print("Gold data saved successfully:")
print(GOLD_PATH)

In [0]:
Testing 

In [0]:
%python
#Create the test dataset

test_data = [
    (
        "TEST_BAD_COORD",
        "Test Bad Coordinates",
        "Los Angeles",
        "CA",
        250.0,       # INVALID longitude
        34.05,
        "TEST_RUN"
    ),
    (
        "TEST_MISSING_CITY",
        "Test Missing City",
        None,        # MISSING CITY
        "CA",
        -118.24,
        34.05,
        "TEST_RUN"
    ),
    (
        "TEST_CLEAN",
        "Test Clean Chapter",
        "Los Angeles",
        "CA",
        -118.24,
        34.05,
        "TEST_RUN"
    )
]

test_df = spark.createDataFrame(
    test_data,
    [
        "chapter_id",
        "chapter_name",
        "city",
        "state",
        "longitude",
        "latitude",
        "ingest_run_id"
    ]
)

display(test_df)

#Apply the coordinate DQ rule

test_invalid_df = (
    test_df
    .filter(coordinate_invalid_condition)
)

display(test_invalid_df)
print("Quarantine test count:", test_invalid_df.count())

#Check Valid records
test_valid_df = (
    test_df
    .filter(~coordinate_invalid_condition)
)

display(test_valid_df)

# Apply city warning rule
test_dq_df = (
    test_valid_df
    .withColumn(
        "dq_status",
        F.when(
            missing_city_condition,
            F.lit("WARNING")
        ).otherwise(
            F.lit("OK")
        )
    )
    .withColumn(
        "dq_warnings",
        F.when(
            missing_city_condition,
            F.array(F.lit("MISSING_OR_UNKNOWN_CITY"))
        ).otherwise(
            F.array().cast("array<string>")
        )
    )
)

display(test_dq_df)



Automated DQ tests

In [0]:
%python
# Test 1: Invalid coordinates must be quarantined

bad_coord_count = test_invalid_df.count()

assert bad_coord_count == 1, (
    f"Expected 1 invalid-coordinate record, "
    f"but found {bad_coord_count}"
)

print("PASS: Invalid coordinate record detected.")

# Test 2: Invalid coordinates must not enter valid Silver

valid_bad_coord_count = (
    test_valid_df
    .filter(F.col("chapter_id") == "TEST_BAD_COORD")
    .count()
)

assert valid_bad_coord_count == 0, (
    "FAIL: Invalid coordinate record entered valid Silver."
)

print("PASS: Invalid coordinate record excluded from Silver.")

# Test 3: Missing city must generate WARNING

warning_count = (
    test_dq_df
    .filter(
        (F.col("chapter_id") == "TEST_MISSING_CITY") &
        (F.col("dq_status") == "WARNING")
    )
    .count()
)

assert warning_count == 1, (
    "FAIL: Missing-city record did not receive WARNING."
)

print("PASS: Missing city correctly marked as WARNING.")

# Test 4: Warning reason must be present

warning_reason_count = (
    test_dq_df
    .filter(
        (F.col("chapter_id") == "TEST_MISSING_CITY") &
        F.array_contains(
            F.col("dq_warnings"),
            "MISSING_OR_UNKNOWN_CITY"
        )
    )
    .count()
)

assert warning_reason_count == 1, (
    "FAIL: MISSING_OR_UNKNOWN_CITY warning not found."
)

print("PASS: Correct warning reason found.")

# Test 5: Clean record must be OK

clean_count = (
    test_dq_df
    .filter(
        (F.col("chapter_id") == "TEST_CLEAN") &
        (F.col("dq_status") == "OK")
    )
    .count()
)

assert clean_count == 1, (
    "FAIL: Clean record is not marked as OK."
)

print("PASS: Clean record correctly marked as OK.")

# Test 6: Production Gold must contain no quarantined records

gold_read_df = spark.read.parquet(GOLD_PATH)
gold_bad_coord_count = (
    gold_read_df
    .filter(
        (F.col("longitude") < -180) |
        (F.col("longitude") > 180) |
        (F.col("latitude") < -90) |
        (F.col("latitude") > 90) |
        F.col("longitude").isNull() |
        F.col("latitude").isNull()
    )
    .count()
)

assert gold_bad_coord_count == 0, (
    "FAIL: Invalid coordinate record found in Gold."
)

print("PASS: Gold contains no invalid-coordinate records.")

In [0]:
%python
test_gold_df = (
    test_dq_df
    .select(
        "chapter_id",
        "chapter_name",
        "city",
        "state",
        "longitude",
        "latitude",
        "dq_status",
        "dq_warnings"
    )
)

# Bad coordinate must NOT reach Gold
assert test_gold_df.filter(
    F.col("chapter_id") == "TEST_BAD_COORD"
).count() == 0, (
    "FAIL: Bad-coordinate record entered Gold."
)

# Missing city MUST reach Gold with WARNING
assert test_gold_df.filter(
    (F.col("chapter_id") == "TEST_MISSING_CITY") &
    (F.col("dq_status") == "WARNING") &
    F.array_contains(
        F.col("dq_warnings"),
        "MISSING_OR_UNKNOWN_CITY"
    )
).count() == 1, (
    "FAIL: Warning record did not reach Gold correctly."
)

# Clean record MUST reach Gold with OK
assert test_gold_df.filter(
    (F.col("chapter_id") == "TEST_CLEAN") &
    (F.col("dq_status") == "OK")
).count() == 1, (
    "FAIL: Clean record did not reach Gold correctly."
)

print("PASS: DQ fixture tests completed successfully.")
print("PASS: Bad coordinates excluded from Gold.")
print("PASS: Missing-city warning published to Gold.")
print("PASS: Clean record published to Gold.")